# External transfer: TCGA-trained model on a GEO cohort

This notebook trains an `E2MModel` on TCGA lung adenocarcinoma (`LUAD`) and applies it to **GSE31210** (Okayama et al.), an independent lung-adenocarcinoma microarray cohort the model has never seen, to demonstrate cross-cohort prediction end to end.

**Requirements:** network access and `GEOparse` for the GEO download:

```bash
pip install GEOparse
```

**Caveat:** GSE31210 is Affymetrix microarray data, on a different measurement scale from the TCGA STAR counts the model is trained on. This notebook shows the *mechanics* of external prediction — align genes by symbol, then predict. For quantitative transfer, the manuscript batch-corrects each external cohort against its TCGA training set (ComBat / rank normalization); that correction is **not** performed here.

In [ ]:
from pathlib import Path
import pandas as pd

from e2m import Dataset, E2MModel

DATA_DIR = Path("e2m_data")
RESULT_DIR = Path("results/external_gse31210")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Train the mutation model on TCGA-LUAD

This downloads and prepares TCGA-LUAD on the first run (cached under `DATA_DIR`) and fits the multitask network on every matched sample. TMB is not needed here, so it is skipped.

In [ ]:
tcga = Dataset.from_tcga(["LUAD"], data_dir=DATA_DIR, with_tmb=False)
model = E2MModel().fit(tcga)
len(tcga.expression), len(model.targets)

## 2. Download and prepare the external cohort

`GEOparse` downloads the series matrix and its platform annotation. We map probes to gene symbols (taking the first symbol of multi-mapping probes), average duplicate symbols, and return a samples-by-genes frame with the same orientation the model expects.

In [ ]:
def _symbol_column(annotation_table):
    for name in ("Gene Symbol", "Gene symbol", "GENE_SYMBOL", "Symbol", "gene_assignment"):
        if name in annotation_table.columns:
            return name
    raise ValueError(
        f"No gene-symbol column in the platform annotation: {list(annotation_table.columns)}"
    )


def load_gse31210(cache_dir):
    """Download GSE31210, map probes to symbols, return a samples-by-genes frame."""
    import GEOparse

    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    gse = GEOparse.get_GEO(geo="GSE31210", destdir=str(cache_dir), silent=True)
    probes = gse.pivot_samples("VALUE")  # probes x samples

    platform = list(gse.gpls.values())[0]
    annotation = platform.table.set_index("ID")
    symbols = annotation[_symbol_column(annotation)].reindex(probes.index).astype(str)
    symbols = symbols.str.split(r"\s*///\s*").str[0].str.strip()

    keep = symbols.notna() & ~symbols.isin({"", "nan", "---"})
    probes = probes.loc[keep]
    probes.index = symbols[keep].values
    expression = probes.apply(pd.to_numeric, errors="coerce").groupby(level=0).mean().T
    expression.index.name = "sample"
    return expression

In [ ]:
external = load_gse31210(DATA_DIR / "geo")
external.shape

## 3. Predict mutation probabilities on the external cohort

Input genes are aligned to the model's training features by symbol. We set `min_feature_overlap=0.0` because a microarray covers only part of the protein-coding transcriptome; missing features are filled with their training means. Because the scales differ and no batch correction is applied, read the numbers as a mechanics demonstration rather than a calibrated result.

In [ ]:
probabilities = model.predict(external, min_feature_overlap=0.0)
probabilities.to_csv(RESULT_DIR / "gse31210_probabilities.csv")
probabilities.iloc[:5, :8]

In [ ]:
probabilities.mean().sort_values(ascending=False).head(10)